# 03 - Cleaning & Feature Engineering

## Objective
Turn the raw application table into a modelling frame: drop duplicates, null the
impossible values, and add ratio features that express credit risk in the way a
credit analyst would.

**Leakage rule applied here:** every engineered column is a function of a single
customer's own attributes. No column statistic (mean, quantile, target encoding)
is used, so these transformations are safe to run before the split. Everything
that *does* learn from the data -- imputation, scaling, encoding -- lives inside
the scikit-learn pipeline in notebook 05.

In [1]:
import sys
from pathlib import Path

# Make `src` importable when the notebook is run from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import pandas as pd

from src import config
from src.data import preprocessing
from src.features.engineering import FEATURE_DOCS, add_credit_risk_features, get_feature_lists

## Data loading and cleaning

In [3]:
raw = preprocessing.load_raw_data()
TARGET = preprocessing.detect_target(raw)
clean, report = preprocessing.clean_data(raw, TARGET)
report

{'input_rows': 32581,
 'duplicates_removed': 165,
 'rows_dropped_invalid_target': 0,
 'invalid_values_nulled': {'person_age': 5,
  'person_emp_length': 2,
  'person_income': 1},
 'id_columns_dropped': [],
 'output_rows': 32416,
 'missing_after_clean': {'person_age': 5,
  'person_income': 1,
  'person_emp_length': 889,
  'loan_int_rate': 3095}}

### Cleaning decisions

1. **Exact duplicates removed.** Identical rows are almost certainly repeated
   records; keeping them would over-weight those borrowers.
2. **Impossible values nulled, not clipped.** An age of 144 is not a real
   extreme value, it is an error; turning it into NaN lets the imputer handle it
   honestly instead of inventing a boundary value.
3. **Missing values left as NaN here.** They are imputed inside the pipeline
   (median for numeric, most frequent for categorical) so the imputation values
   are learned from the training fold only.
4. **Rows with an unusable target dropped.** A label cannot be imputed.
5. **Genuine outliers kept.** High incomes and large loans are real and informative.

In [4]:
clean.isna().sum().to_frame("missing_after_clean")

,missing_after_clean
person_age,5
person_income,1
person_home_ownership,0
person_emp_length,889
loan_intent,0
loan_grade,0
loan_amnt,0
loan_int_rate,3095
loan_status,0
loan_percent_income,0


## Feature engineering

In [5]:
featured = add_credit_risk_features(clean)
new_columns = [c for c in featured.columns if c not in clean.columns]
pd.DataFrame({"feature": new_columns,
              "rationale": [FEATURE_DOCS.get(c, "") for c in new_columns]})

,feature,rationale
0,loan_to_income_ratio,Loan amount / annual income. Core affordabilit...
1,annual_interest_cost,loan_amnt * interest rate. Absolute yearly cos...
2,est_monthly_payment,Amortised monthly instalment assuming a 60-mon...
3,interest_to_income,Annual interest cost / income. Interest burden...
4,payment_to_income,Estimated annual instalments / income. Debt-se...
5,credit_history_ratio,Credit-history length / years since age 18. Ca...
6,employment_stability,Employment length / years since age 18. Share ...
7,income_per_credit_year,Income / (1 + credit history length). Distingu...
8,log_income,log1p(income). Tames the strong right skew for...
9,has_prior_default,1 if the credit bureau file records a previous...


In [6]:
featured[new_columns].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
loan_to_income_ratio,32415.0,0.171,0.107,0.001,0.090,0.148,0.229,0.830
annual_interest_cost,29321.0,1086.380,879.010,48.800,475.615,820.000,1423.200,7087.500
est_monthly_payment,32416.0,205.660,141.437,8.333,103.333,169.976,268.026,932.161
interest_to_income,29320.0,0.019,0.014,0.000,0.009,0.015,0.026,0.115
payment_to_income,32415.0,0.044,0.028,0.000,0.022,0.037,0.059,0.210
credit_history_ratio,32411.0,0.610,0.217,0.250,0.444,0.583,0.750,1.500
employment_stability,31522.0,0.606,0.492,0.000,0.176,0.500,1.000,1.500
income_per_credit_year,32415.0,12156.361,9356.907,168.552,5891.625,9980.000,15800.000,141666.667
log_income,32415.0,10.926,0.565,8.294,10.559,10.915,11.280,14.528
has_prior_default,32416.0,0.177,0.381,0.000,0.000,0.000,0.000,1.000


### Do the new features separate the classes?

In [7]:
engineered_numeric = [c for c in new_columns if pd.api.types.is_numeric_dtype(featured[c])]
featured.groupby(TARGET)[engineered_numeric].mean().T.round(3)

loan_status,0,1
loan_to_income_ratio,0.149,0.249
annual_interest_cost,985.630,1443.771
est_monthly_payment,195.230,242.924
interest_to_income,0.016,0.032
payment_to_income,0.038,0.066
credit_history_ratio,0.608,0.616
employment_stability,0.623,0.544
income_per_credit_year,12954.559,9304.736
log_income,11.010,10.623
has_prior_default,0.141,0.306


### Leakage self-check

* No engineered feature reads the target.
* No engineered feature uses another row's values.
* `loan_grade` and `loan_int_rate` are the lender's own risk assessment. They are
  legitimate inputs (known at decision time) but they make the model partly a
  *re-learner of the lender's policy* -- flagged in the README limitations.

In [8]:
assert TARGET not in [c for c in new_columns]
numeric, categorical = get_feature_lists(featured, TARGET,
                                         drop=["cb_person_default_on_file"])
print(f"{len(numeric)} numeric features:\n{numeric}\n")
print(f"{len(categorical)} categorical features:\n{categorical}")

17 numeric features:
['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'loan_to_income_ratio', 'annual_interest_cost', 'est_monthly_payment', 'interest_to_income', 'payment_to_income', 'credit_history_ratio', 'employment_stability', 'income_per_credit_year', 'log_income', 'has_prior_default']

6 categorical features:
['person_home_ownership', 'loan_intent', 'loan_grade', 'age_group', 'income_group', 'loan_amount_group']


## Results

In [9]:
preprocessing.save_dataframe(featured, config.FEATURED_DATA_FILE)
featured.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,...,interest_to_income,payment_to_income,credit_history_ratio,employment_stability,income_per_credit_year,log_income,has_prior_default,age_group,income_group,loan_amount_group
0,22.0,59000.0,RENT,NaN,PERSONAL,D,35000,16.02,1,0.59,...,0.095034,0.173187,0.750000,NaN,14750.000000,10.985310,1,18-25,30k-60k,25k+
1,21.0,9600.0,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,...,0.011604,0.027265,0.666667,1.500000,3200.000000,9.169623,0,18-25,<30k,<5k
2,25.0,9600.0,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,...,0.073734,0.155970,0.428571,0.142857,2400.000000,9.169623,0,18-25,<30k,5k-10k
3,23.0,65500.0,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,...,0.081382,0.153321,0.400000,0.800000,21833.333333,11.089821,0,18-25,60k-100k,25k+
4,24.0,54400.0,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,...,0.091811,0.180727,0.666667,1.333333,10880.000000,10.904138,1,18-25,30k-60k,25k+


## Interpretation
The engineered set is deliberately small and interpretable: affordability ratios,
stability ratios and business-defined bands. Each one can be explained to a credit
officer in a single sentence.

## Conclusion
`data/processed/credit_risk_featured.csv` is the input for the SQL analytics
(notebook 04) and the modelling (notebook 05).